## Приоритезация обращений. Model

**Подход к решению**:
1) провел EDA (см. файл `eda.ipynb` - там есть все выводы)
2) валидация скользящим expanding-окном по дням, каждый фолд разделен на train/val/test; val (2 дня) - под early stopping и подбор параметров, test (3 дня) - для оценки моделей
3) модель - Catboost, параметры подбирал с optuna - сравнивал с logReg, CatboostRanker, пробовал реализовать бэггинг над используемыми моделями, ранжирующий ответы - прироста не получил
4) с помощью `events.csv` провел feature engineering

In [ ]:
# подхватывать правки локальных модулей без рестарта ядра
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier, CatBoostRanker
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# локальные модули
import preprocessor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [65]:
ROOT = Path(".")
DATA_DIR = ROOT / "data"

TARGET = "target"
RANDOM_STATE = 42

TEST_DAYS = 3  # test
VALID_DAYS = 2  # val
N_FOLDS = 3  # число фолдов
MIN_TRAIN_DAYS = 5  # минимальный размер train без val

SEEDS = (0, 1, 2)  # для усреднения по сидам
TUNE_SEEDS = (0, 1)

### Загрузка данных

Загружаем обучающую выборку, тестовую выборку и события.

In [66]:
train = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test_df:", test_df.shape)
print("events:", events.shape)

train: (13694, 119)
test_df: (4306, 118)
events: (254705, 7)


### Выбор признаков

`Preprocessor` из `preprocessor.py`:

- **добавляет 35 признаков из `events.csv`** с обязательным фильтром `event_ts < assignment_ts` — 8.17% приджойненных событий лежат после назначения и описывают будущий исход, использовать их запрещено;

- **6 базовых агрегатов** по всей истории лида: `ev_n`, `ev_recency_h` (свежесть последнего действия), `ev_span_h` (растянутость активности), `ev_slot_nuniq`, `ev_ctx_nuniq`, `ev_type_nuniq`;

- **29 оконных признаков** (окна 1/3/7 дней): счётчики по типам событий (5 × 3), `ev_hi_intent_{w}d` (favorite + chat_open + call_click), `ev_n_{w}d`, счётчики по 8 контекстам показа. Считаются одним `groupby` через флаги принадлежности окну;

- **`NaN` и `0` разведены**: у лида есть события, но в окне их нет => `0`; лида нет в логе вовсе (20 из 13694, 0.15%) => `NaN` по всем `ev_*`. Catboost обрабатывает `NaN` нативно;

- **убирает дубли**: `is_weekend` (V Крамера 1.00 с `assignment_weekday`) и `price_bucket` (correlation ratio 0.81 с `item_price_log`) — EDA1.2;

**Эвристика**: действия более серьезного намерения - т.е. не просто просмотр или поиск - а именно связь/сохранение в избранные. Так вот такие действия должны по идее сильнее влиять на таргет - а их сумма тем более. --- **получилось улучшить скор**

### Валидация

`test_df` лежит строго после `train` по времени:
- `train 07.04-22.04 = 16 дней`
- `test_df 23.04-27.04 = 5 дней`

#### Метрики

In [67]:
def daily_ap(y_true, scores, dates) -> float:
    """Метрика соревнования: AP считается внутри каждого дня и усредняется по дням.

    Дни без единого положительного примера пропускаем - AP на них не определён.
    """
    df = pd.DataFrame({"y": y_true, "s": scores, "d": dates})
    per_day = [average_precision_score(g["y"], g["s"])
               for _, g in df.groupby("d") if g["y"].sum() > 0]
    return float(np.mean(per_day))

#### Схема

Обычная кросс-валидация не подходит. Двигаемся скользящим окном по дням, чтобы не было data leak. **Минимальный квант --- 1 день**, т.к. метрика Daily AP.

Фолд --- это **тройка подряд идущих блоков** `train | valid | test`, которая едет от конца выборки к началу. `valid` стоит вплотную перед `test`: на нём подбираются гиперпараметры этого же фолда, и подбор всегда смотрит на дни, непосредственно предшествующие замеру, --- ровно так же, как перед сабмитом.

```
fold 0: train 07.04-17.04 (11д) | valid 18-19 | test 20-22
fold 1: train 07.04-14.04  (8д) | valid 15-16 | test 17-19
fold 2: train 07.04-11.04  (5д) | valid 12-13 | test 14-16
```

Шаг тройки равен `TEST_DAYS`, поэтому **тест-блоки не перекрываются**: три замера сделаны на разных днях и их можно усреднять. val-блоки тоже не пересекаются с тестами своего фолда => параметры подбираются честно.

**Expanding, а не rolling**: данных мало, роскоши дропать начало выборки нет => абсолютные скоры фолдов между собой несравнимы, поэтому решения принимаем по разностям моделей внутри фолда, а не по разнице средних.

In [68]:
def folds(df, valid_days=VALID_DAYS, test_days=TEST_DAYS, n_folds=N_FOLDS, min_train_days=MIN_TRAIN_DAYS, step_days=None):
    """
    Тройки (train, valid, test) из подряд идущих дней, едущие к началу выборки.
    
    - step_days=None ---> шаг равен длине тест-блока, то есть тесты не перекрываются.
    """
    step_days = step_days or test_days
    days = pd.to_datetime(df["assignment_date"]).dt.date
    ordered = sorted(days.unique())

    for k in range(n_folds):
        test_end = len(ordered) - k * step_days
        test_start = test_end - test_days
        valid_start = test_start - valid_days

        if valid_start < min_train_days:
            raise ValueError(f"фолд {k} не помещается: на train осталось бы {valid_start}д "
                             f"при минимуме {min_train_days}")

        yield (df[days < ordered[valid_start]],
               df[days.isin(ordered[valid_start:test_start])],
               df[days.isin(ordered[test_start:test_end])])


show = lambda df: sorted(pd.to_datetime(df.assignment_date).dt.date.unique())

for i, (tr, val, te) in enumerate(folds(train)):
    total = len(tr) + len(val) + len(te)
    part = lambda df, label: (f"{label} {show(df)[0]}..{show(df)[-1]} "
                              f"({len(show(df))}д, {len(df)} строк, {len(df) / total:.0%})")
    print(f"fold {i}: {part(tr, 'train')} | {part(val, 'valid')} | {part(te, 'test')}")

fold 0: train 2026-04-07..2026-04-17 (11д, 9397 строк, 69%) | valid 2026-04-18..2026-04-19 (2д, 1738 строк, 13%) | test 2026-04-20..2026-04-22 (3д, 2559 строк, 19%)
fold 1: train 2026-04-07..2026-04-14 (8д, 6743 строк, 61%) | valid 2026-04-15..2026-04-16 (2д, 1753 строк, 16%) | test 2026-04-17..2026-04-19 (3д, 2639 строк, 24%)
fold 2: train 2026-04-07..2026-04-11 (5д, 4255 строк, 50%) | valid 2026-04-12..2026-04-13 (2д, 1646 строк, 19%) | test 2026-04-14..2026-04-16 (3д, 2595 строк, 31%)


[идея] возможно есть смысл пересчитать запуски `folds` для разные значения размеров valid, test и усреднить

### Модель

#### Модель (#0.Baseline LogReg + #1. LogReg)

Используем простую Logistic Regression:

- числовые признаки: заполнение пропусков медианой и scaling;
- категориальные признаки: заполнение самым частым значением и one-hot encoding;
- `class_weight="balanced"` из-за дисбаланса классов.

`baseline` без изменений из `quickstart.ipynb`, `logreg` имеет препроцессинг признаков 

In [69]:
def build_baseline(seed=RANDOM_STATE, **params):
    """Логрег детерминирован, сид на него не влияет - но интерфейс общий с бустингом."""
    return preprocessor.Pipeline(
        events=events,
        baseline=True,
        model=LogisticRegression(random_state=seed, max_iter=1000,
                                 class_weight="balanced", **params),
    )

In [70]:
def build_logreg(seed=RANDOM_STATE, **params):
    """Логрег детерминирован, сид на него не влияет - но интерфейс общий с бустингом."""
    return preprocessor.Pipeline(
        events=events,
        model=LogisticRegression(random_state=seed, max_iter=1000,
                                 class_weight="balanced", **params),
    )

#### Модель (#2. Catboost)

Логрегрессии нужен полный препроцессинг, бустингу он наоборот вредит. Поэтому CatBoost собираем с `sklearn_preprocess=False`, то есть без `ColumnTransformer`:

- **пропуски не импутирую** --- CatBoost обрабатывает `NaN` нативно, а замена медианой стёрла бы саму информацию о пропуске
- **не делаю One-Hot** --- вместо него встроенная обработка категориальных признаков
- `boosting_type="Plain"` вместо `Ordered`, который CatBoost включает сам на малых выборках --- быстрее и не хуже
- `class_weights=balanced` из-за разброса
- метрику решил смотреть `AUC`

In [71]:
def build_catboost(seed=RANDOM_STATE, **params):
    """Catboost без sklearn-препроцессинга. params переопределяют дефолты."""
    defaults = dict(
        iterations=3000,
        early_stopping_rounds=100,
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        cat_features=preprocessor.CATEGORICAL_COLUMNS,  # NaN в категориальных нет => ошибки не будет
        boosting_type="Plain",
        verbose=False,
    )
    return preprocessor.Pipeline(
        events=events,
        sklearn_preprocess=False,
        model=CatBoostClassifier(random_seed=seed, **{**defaults, **params}),
    )


#### Модель (#4. CatBoostRanker)

Метрика соревнования --- Daily AP, то есть **ранжирование внутри дня**, а классификатор учится Logloss'ом, то есть калибровке вероятностей. Ранжирующий лосс оптимизирует ровно то, что меряется.

Группа для ранжирования = день назначения, потому что группа метрики и группа обучения тогда совпадают. Группы для этого подходят - около 800-900 лид в день, доля 1 стабильно около 0.2.

`group_id` sklearn-пайплайн прокинуть не умеет, поэтому в `preprocessor.Pipeline` заведён флаг `ranker=True`: он собирает группы из `assignment_date`, сортирует строки по группе (требование CatBoost) и подменяет `predict_proba`, т.к. ранкер выдаёт произвольные скоры, а не вероятности.

In [72]:
def build_ranker(seed=RANDOM_STATE, **params):
    """CatBoostRanker: учим порядок внутри дня вместо вероятностей.

    eval_metric задаём явно: без неё ранняя остановка идёт по самому лоссу, а он у
    YetiRank стохастический (пересемплируется каждую итерацию) и не растёт монотонно -
    детектор срабатывает на 2-м дереве, AP падает до 0.48 вместо 0.60.

    auto_class_weights тут нет: ранжирующему лоссу дисбаланс не мешает, он сравнивает
    объекты внутри группы, а не предсказывает абсолютную вероятность.
    """
    defaults = dict(
        iterations=3000,
        early_stopping_rounds=100,
        loss_function="QuerySoftMax",
        eval_metric="PRAUC",
        cat_features=preprocessor.CATEGORICAL_COLUMNS,
        boosting_type="Plain",
        verbose=False,
    )
    return preprocessor.Pipeline(
        events=events,
        sklearn_preprocess=False,
        ranker=True,
        model=CatBoostRanker(random_seed=seed, **{**defaults, **params}),
    )


#### Модель (#5. Бэггинг Catboost + LightGBM/XGBoost)

**Идея**: данные фреймворки по разному строят деревья при реализации GB => их аггрегация может дать лучший результат (НО для вторых надо OneHot, получим разные пространства признаков)

**ВЫВОД** - ожидаемый прирост не сильно большой (порядка 0.001) - важнее сконцентрироваться на FE

#### Подбор гиперпараметров (optuna + Catboost)

Подбор только для `CatBoost`. 

TPE, а не сетка: пространство непрерывное.

`tune` принимает `train` и `valid` **одного фолда**, поэтому у каждого фолда набор параметров свой + усредняем значение по сидам.

In [73]:
def fit_score(build_model, train_df, test_df, seeds=SEEDS, val_df=None) -> tuple[list, list]:
    """
    В модель передаём строки целиком, а не срез по признакам: препроцессору нужны
    lead_id и assignment_ts, чтобы приджойнить события до момента назначения

    val_df нужен бустингам для early_stopping, логрегам - нет

    Возвращает метрики по сидам и число деревьев по сидам: на скольких итерациях
    остановился early stopping, нужно потом для финальной модели
    """
    eval_set = (val_df, val_df[TARGET]) if val_df is not None else None
    scores, trees = [], []
    for seed in seeds:
        model = build_model(seed).fit(train_df, train_df[TARGET], eval_set=eval_set)
        proba = model.predict_proba(test_df)[:, 1]
        scores.append(daily_ap(test_df[TARGET], proba, test_df["assignment_date"]))
        trees.append(getattr(model.model, "tree_count_", np.nan))  # у логрега его нет
    return scores, trees

In [83]:
N_TRIALS = 20
optuna.logging.set_verbosity(optuna.logging.WARNING)

def suggest_params(trial) -> dict:
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
    }

def tune(train_part, valid_part, n_trials=N_TRIALS, seeds=TUNE_SEEDS):
    """
    optuna на valid-блоке одного фолда. Возвращает study целиком: best_params
    идут в замер, а разброс лучших конфигураций между фолдами - в диагностику
    
    на valid части обучаются одновременно и параметры, и early_stopping,
    чтобы не дообучаться на valid перед замером
    """
    def objective(trial):
        params = suggest_params(trial)
        build = lambda seed: build_catboost(seed, **params)
        scores, _ = fit_score(build_model=build, train_df=train_part, test_df=valid_part,
                              val_df=valid_part, seeds=seeds)
        return float(np.mean(scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=n_trials,  show_progress_bar=True)
    return study

#### Сравнение результатов моделей

Один проход по фолдам: на каждом подбираем параметры для кэтбуста на его `val`, обучаемся на `train` (`val` для CatBoost остаётся честным eval_set под early stopping, поэтому в обучение не попадает) и меряем все модели на его `test`.

Ориентир из бейзлайна (`make_validation_split` в `quickstart.ipynb`, обычный split 80/20 по датам): AP **0.48506**. Напрямую не сравнить --- другая схема валидации, но совпадает `baseline` модель с `quickstart.ipynb`.

In [91]:
def walk_forward(df, n_trials=N_TRIALS) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows, tree_rows, chosen = [], [], {}

    for k, (tr, val, te) in enumerate(folds(df)):
        print(f"fold {k}: test {show(te)[0]}..{show(te)[-1]} ", end=" ")

        study = tune(tr, val, n_trials=n_trials)
        chosen[f"fold {k}"] = study.best_params

        # дообучаться на valid перед замером нельзя - т.к. val передается в eval_set
        # train_df = pd.concat([tr, val])
        builders = {
            "baseline": build_baseline,
            "logreg": build_logreg,
            "catboost_default": build_catboost,
            "catboost_tuned": lambda seed: build_catboost(seed, **study.best_params),
            "ranker_qsm": build_ranker,
            "ranker_yeti": lambda seed: build_ranker(seed, loss_function="YetiRank", eval_metric="NDCG"),
        }
        for name, build in builders.items():
            # eval_set (early stopping) нужен бустингам, логрегам - нет
            val_kwarg = None if name in ("baseline", "logreg") else val
            scores, trees = fit_score(build_model=build, train_df=tr, test_df=te, val_df=val_kwarg)

            for score in scores:
                rows.append({"fold": k, "model": name, "daily_ap": score})
            for n in trees:
                tree_rows.append({"fold": k, "model": name, "trees": n, "n_train": len(tr)})

        print(f"на valid best={study.best_value:.4f}")

    return pd.DataFrame(rows), pd.DataFrame(tree_rows), pd.DataFrame(chosen).T

In [92]:
print(f"...", flush=True)
results, trees_df, best_params = walk_forward(train)

per_fold = results.groupby(["fold", "model"])["daily_ap"].mean().unstack()
display(per_fold.round(4))
display(per_fold.agg(["mean", "std"]).round(4))

# шум: разброс по сидам внутри фолда - ориентир, 
# какую разницу моделей можно считать значимой
print(f"шум (max sd по сидам внутри фолда): "
      f"{results.groupby(['fold','model'])['daily_ap'].std().max():.4f}")

...
fold 0: test 2026-04-20..2026-04-22  

  0%|          | 0/20 [00:00<?, ?it/s]

на valid best=0.6963
fold 1: test 2026-04-17..2026-04-19  

  0%|          | 0/20 [00:00<?, ?it/s]

на valid best=0.6462
fold 2: test 2026-04-14..2026-04-16  

  0%|          | 0/20 [00:00<?, ?it/s]

на valid best=0.6382


model,baseline,catboost_default,catboost_tuned,logreg,ranker_qsm,ranker_yeti
fold,,,,,,
0,0.4928,0.6790,0.6798,0.6545,0.6725,0.5974
1,0.4635,0.6819,0.6881,0.6621,0.6697,0.6286
2,0.4184,0.6229,0.6249,0.6078,0.5984,0.5452


model,baseline,catboost_default,catboost_tuned,logreg,ranker_qsm,ranker_yeti
mean,0.4582,0.6612,0.6643,0.6414,0.6468,0.5904
std,0.0375,0.0332,0.0343,0.0294,0.0420,0.0422


шум (max sd по сидам внутри фолда): 0.0137


### Submission

Обучаем модель на всем train и строим score для test_df.
Файл для отправки должен содержать две колонки: `lead_id` и `score`.

выбор финальной модели исходя из результатов обучения

In [93]:
FINAL_MODEL = "catboost_tuned"

# Параметры берём с фолда 0, поэтому и число итераций - оттуда же: у другого фолда
# другой learning_rate, а значит и другая точка остановки. Фолд 0 еще самый
# большой по train, то есть ближе всех к обучению на полном train.
fold0 = trees_df[(trees_df.model == FINAL_MODEL) & (trees_df.fold == 0)]
assert len(fold0), f"{FINAL_MODEL} нет в trees_df: {sorted(trees_df.model.unique())}"

FINAL_ITERATIONS = int(round(fold0.trees.median() * len(train) / fold0.n_train.iloc[0]))
print(f"фолд 0: деревьев {fold0.trees.median():.0f} на {fold0.n_train.iloc[0]} строках "
      f"-> iterations={FINAL_ITERATIONS} на {len(train)} строках")


фолд 0: деревьев 1489 на 9397 строках -> iterations=2170 на 13694 строках


In [96]:
final_params = {**best_params.loc["fold 0"].to_dict(), "iterations": FINAL_ITERATIONS} \
    if len(best_params) else {"iterations": FINAL_ITERATIONS}

builders = {
    "baseline": build_baseline,
    "logreg": build_logreg,
    "catboost_tuned": lambda seed: build_catboost(seed, **final_params),
    "ranker_qsm": lambda seed: build_ranker(seed, **final_params),
}
final_params

{'learning_rate': 0.02086271874273327,
 'depth': 5.0,
 'l2_leaf_reg': 9.307352594320825,
 'iterations': 2170}

In [97]:
final_model = builders[FINAL_MODEL](RANDOM_STATE).fit(train, train[TARGET])

In [98]:
submission = pd.DataFrame({
    "lead_id": test_df["lead_id"].astype(str),
    "score": final_model.predict_proba(test_df)[:, 1],
})
submission.to_csv(ROOT / "submission.csv", index=False)
submission.head()

,lead_id,score
0,lead_97e409eb8f8c8246,0.345721
1,lead_55310edb4489f9e9,0.346110
2,lead_e7f653a2c6a7eee8,0.975369
3,lead_22f8e1cfc487ac20,0.079844
4,lead_48b638b839abfac3,0.261638


In [99]:
# Минимальные проверки формата перед загрузкой.
assert list(submission.columns) == ["lead_id", "score"]
assert len(submission) == len(test_df)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()

print("submission.csv is ready")

submission.csv is ready
